### 🧠 What is a WGAN?

A Wasserstein GAN is a special kind of GAN (Generative Adversarial Network) designed to make training more stable and produce better quality outputs (like images).

🧩 Real-World Analogy

Imagine:

-	A forger (generator) is trying to create fake paintings that look real.
-	An art critic (discriminator) is trying to judge how real or fake each painting is.

In classic GANs, the critic says “real” or “fake” — like a binary answer. But this often leads to fights and instability.

In WGAN, the critic doesn’t say yes/no. Instead, they give a score — like:

“This painting looks 70% real.”

This smooth scoring system helps the forger improve gradually, rather than guessing randomly.

✅ Why WGAN is Better:
-	More stable training (less likely to crash).
-	Avoids mode collapse (where generator produces only one kind of output).
-	Gives useful loss values (loss actually means something, unlike in normal GANs).

# Wasserstein GAN (WGAN) using the MNIST dataset

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Reshape, Flatten, LeakyReLU, Input
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.datasets import mnist
import matplotlib.pyplot as plt

# Set seed
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
(X_train, _), (_, _) = mnist.load_data()
X_train = X_train / 127.5 - 1.0  # Normalize to [-1, 1]
X_train = X_train.reshape(-1, 28 * 28)
img_shape = (28 * 28,)
latent_dim = 100

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
def build_generator():
    model = Sequential()
    model.add(Dense(128, input_dim=latent_dim))
    model.add(LeakyReLU(0.2))
    model.add(Dense(784, activation='tanh'))
    model.add(Reshape((784,)))
    return model

In [ ]:
def build_critic():
    model = Sequential()
    model.add(Dense(128, input_dim=784))
    model.add(LeakyReLU(0.2))
    model.add(Dense(1))  # No sigmoid
    return model

In [ ]:
# Optimizer
optimizer = tf.keras.optimizers.RMSprop(learning_rate=0.00005)

# Build Critic
critic = build_critic()
critic.compile(loss=lambda y_true, y_pred: tf.reduce_mean(y_true * y_pred), optimizer=optimizer)

# Build Generator
generator = build_generator()

# WGAN combined model
z = Input(shape=(latent_dim,))
img = generator(z)
critic.trainable = False
validity = critic(img)
combined = Model(z, validity)
combined.compile(loss=lambda y_true, y_pred: tf.reduce_mean(y_true * y_pred), optimizer=optimizer)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
def train(epochs, batch_size=64, sample_interval=1000, n_critic=5, clip_value=0.01):
    valid = -np.ones((batch_size, 1))  # real = -1
    fake = np.ones((batch_size, 1))    # fake = 1

    for epoch in range(epochs):
        for _ in range(n_critic):
            # Train Critic
            idx = np.random.randint(0, X_train.shape[0], batch_size)
            real_imgs = X_train[idx]

            noise = np.random.normal(0, 1, (batch_size, latent_dim))
            fake_imgs = generator.predict(noise)

            d_loss_real = critic.train_on_batch(real_imgs, valid)
            d_loss_fake = critic.train_on_batch(fake_imgs, fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # Clip weights
            for layer in critic.layers:
                weights = layer.get_weights()
                weights = [np.clip(w, -clip_value, clip_value) for w in weights]
                layer.set_weights(weights)

        # Train Generator
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        g_loss = combined.train_on_batch(noise, valid)

        # Progress output
        if epoch % 100 == 0:
            print(f"{epoch} [D loss: {d_loss:.4f}] [G loss: {g_loss:.4f}]")

        if epoch % sample_interval == 0:
            sample_images(epoch)



def sample_images(epoch, n=5):
    noise = np.random.normal(0, 1, (n * n, latent_dim))
    gen_imgs = generator.predict(noise)
    gen_imgs = 0.5 * gen_imgs + 0.5  # Rescale to [0, 1]

    fig, axs = plt.subplots(n, n)
    cnt = 0
    for i in range(n):
        for j in range(n):
            axs[i, j].imshow(gen_imgs[cnt].reshape(28, 28), cmap='gray')
            axs[i, j].axis('off')
            cnt += 1
    plt.tight_layout()
    plt.savefig(f"wgan_epoch_{epoch}.png")
    plt.close()

In [ ]:
train(epochs=5001, batch_size=128, sample_interval=1000)

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step  


/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/trainer.py:82: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Streaming output truncated to the last 5000 lines.
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 

KeyboardInterrupt: 